# 🧠 nanoGPT UI — Google Colab

Run the full nanoGPT UI suite directly in Colab.

| Tool | Description | How to access |
|---|---|---|
| **Inference UI** | Text generation (Gradio) | Public `gradio.live` URL (auto-generated) |
| **Training Dashboard** | Real-time loss/LR/MFU charts (Streamlit) | Public `loca.lt` tunnel URL |
| **Model Comparison** | Side-by-side checkpoint outputs (Streamlit) | Public `loca.lt` tunnel URL |
| **Results Exporter** | Export perplexity/stats to JSON/CSV/Markdown | Run inline |

> **Run cells top-to-bottom.** Each section is independent after setup.

## 1️⃣ Setup — Clone repo & install dependencies

In [ ]:
# Clone the nanoGPT repo (skip if already cloned)
import os

REPO_URL = "https://github.com/cartel-codes/nanoGPT.git"  # ← your repo
REPO_DIR = "/content/nanoGPT"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Install all dependencies
!pip install -q gradio streamlit plotly tiktoken pyngrok
print("✓ All packages installed")

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ (Optional) Upload a checkpoint

If you have a trained `.pt` checkpoint, upload it here.
Otherwise the UI will offer the OpenAI GPT-2 pretrained weights.

In [ ]:
import os
os.makedirs("/content/nanoGPT/out", exist_ok=True)

# Option A: Upload from your computer
from google.colab import files
print("Upload your ckpt.pt file (or cancel to use GPT-2 pretrained weights)")
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f"/content/nanoGPT/out/{fname}"
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"✓ Saved to {dest}")
except Exception:
    print("No file uploaded — will use GPT-2 pretrained weights")

In [ ]:
# Option B: Download checkpoint from Google Drive
# Uncomment and fill in your file ID

# GDRIVE_FILE_ID = "YOUR_FILE_ID_HERE"
# !gdown https://drive.google.com/uc?id={GDRIVE_FILE_ID} -O /content/nanoGPT/out/ckpt.pt
# print("✓ Checkpoint downloaded from Google Drive")

## 3️⃣ Inference UI (Gradio)

Gradio generates a **public `gradio.live` URL** automatically in Colab.
The link will appear in the output below after running the cell.

> ⚠️ The cell will keep running while the UI is active. Click **Stop** (■) to shut it down.

In [ ]:
import os, sys
os.chdir("/content/nanoGPT")
if "/content/nanoGPT" not in sys.path:
    sys.path.insert(0, "/content/nanoGPT")

from ui.app_inference import build_ui

demo = build_ui()
demo.launch(
    share=True,          # ← generates a public gradio.live URL
    show_error=True,
    debug=False,
)

## 4️⃣ Training Dashboard (Streamlit)

Streamlit doesn't have a built-in sharing mechanism, so we expose it via **pyngrok**.

> **First time only:** Sign up at https://dashboard.ngrok.com/signup (free), copy your authtoken, and paste it below.

In [ ]:
# ---- Set your ngrok authtoken (only needed once per Colab session) ----
NGROK_TOKEN = ""  # ← paste your token from https://dashboard.ngrok.com/get-started/your-authtoken

if not NGROK_TOKEN:
    print("⚠️  No ngrok token set — using localtunnel instead (no token needed)")
    USE_NGROK = False
else:
    USE_NGROK = True
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print("✓ ngrok token configured")

In [ ]:
import subprocess, threading, time, os, sys

os.chdir("/content/nanoGPT")
STREAMLIT_PORT = 8501
APP_FILE = "/content/nanoGPT/ui/app_dashboard.py"

# Launch Streamlit in background
proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", APP_FILE,
        "--server.port", str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(4)  # wait for Streamlit to start

# Expose via tunnel
if USE_NGROK:
    from pyngrok import ngrok
    tunnel = ngrok.connect(STREAMLIT_PORT)
    public_url = tunnel.public_url
else:
    # localtunnel fallback (no token needed)
    lt_proc = subprocess.Popen(
        ["npx", "localtunnel", "--port", str(STREAMLIT_PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    time.sleep(3)
    lt_out = lt_proc.stdout.readline().decode().strip()
    # localtunnel prints: your url is: https://...
    public_url = lt_out.split("is: ")[-1] if "is: " in lt_out else lt_out

print()
print("=" * 60)
print(f"🧠 nanoGPT Training Dashboard")
print(f"   Public URL: {public_url}")
print("=" * 60)
print("Keep this cell running. Click Stop (■) to shut down.")

try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print("Streamlit stopped.")

## 5️⃣ Model Comparison (Streamlit)

Same tunnel approach as the dashboard. Run this in a **separate cell** (or after stopping the dashboard).

In [ ]:
import subprocess, time, sys, os

os.chdir("/content/nanoGPT")
COMPARE_PORT = 8502
APP_FILE = "/content/nanoGPT/ui/app_compare.py"

proc_cmp = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", APP_FILE,
        "--server.port", str(COMPARE_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(4)

if USE_NGROK:
    from pyngrok import ngrok
    tunnel_cmp = ngrok.connect(COMPARE_PORT)
    url_cmp = tunnel_cmp.public_url
else:
    lt_cmp = subprocess.Popen(
        ["npx", "localtunnel", "--port", str(COMPARE_PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    time.sleep(3)
    lt_out = lt_cmp.stdout.readline().decode().strip()
    url_cmp = lt_out.split("is: ")[-1] if "is: " in lt_out else lt_out

print()
print("=" * 60)
print(f"⚖️  nanoGPT Model Comparison")
print(f"   Public URL: {url_cmp}")
print("=" * 60)

try:
    proc_cmp.wait()
except KeyboardInterrupt:
    proc_cmp.terminate()
    print("Streamlit comparison stopped.")

## 6️⃣ Results Exporter

Run inline — no tunnel needed. Outputs a JSON, CSV, or Markdown file you can download.

In [ ]:
import os, sys
os.chdir("/content/nanoGPT")
if "/content/nanoGPT" not in sys.path:
    sys.path.insert(0, "/content/nanoGPT")

CHECKPOINT = "out/ckpt.pt"          # ← change if your checkpoint has a different name
LOG_FILE   = "logs/metrics.json"    # ← created automatically by train.py (optional)
FORMAT     = "markdown"             # ← "json" | "csv" | "markdown"
OUTPUT     = "results_report.md"    # ← output filename

if not os.path.exists(CHECKPOINT):
    print(f"⚠️  Checkpoint not found at {CHECKPOINT}")
    print("   Upload a checkpoint in Section 2, or change CHECKPOINT path above.")
else:
    !python ui/results_exporter.py \
        --checkpoint {CHECKPOINT} \
        --log {LOG_FILE} \
        --format {FORMAT} \
        --output {OUTPUT}

    # Display the report in the notebook
    if FORMAT == "markdown" and os.path.exists(OUTPUT):
        from IPython.display import Markdown, display
        display(Markdown(open(OUTPUT).read()))
    elif FORMAT == "json" and os.path.exists(OUTPUT):
        import json
        print(json.dumps(json.load(open(OUTPUT)), indent=2))

In [ ]:
# Download the exported results file
from google.colab import files
import os

if os.path.exists(OUTPUT):
    files.download(OUTPUT)
    print(f"✓ Downloading {OUTPUT}")
else:
    print("No output file to download yet — run the cell above first.")

## 7️⃣ Train with metrics logging

If you want to train a model *in* Colab and monitor it with the dashboard:

```bash
# Tiny Shakespeare dataset (fast, ~5 minutes on T4)
!python data/shakespeare_char/prepare.py
!python train.py config/train_shakespeare_char.py \
    --device=cuda --compile=False \
    --eval_interval=250 --eval_iters=20 \
    --log_interval=10 \
    --n_layer=6 --n_head=6 --n_embd=384 \
    --max_iters=5000 --batch_size=64
```

While it runs, open **Section 4** in a separate browser tab to watch the dashboard update every 10 seconds.